# Normalization / Standardization Workflows

## Definition
**Scaling** numeric features to a common range is essential for many ML algorithms.

### Why Scale?
- **Distance-based models** (k-NN, SVM, k-means) are sensitive to scale
- **Gradient descent** converges faster with scaled features
- **PCA** is dominated by high-variance features without scaling
- **Neural networks** require well-scaled inputs for stable training

### Methods Comparison
| Method | Formula | Range | When to Use |
|--------|---------|-------|-------------|
| **Z-score (Standard)** | `(x - μ) / σ` | ~[-3, 3] | Normal-ish dist, outlier-sensitive |
| **Min-Max** | `(x - min) / (max - min)` | [0, 1] | Fixed range needed |
| **Robust Scaler** | `(x - median) / IQR` | varies | Dataset with outliers |
| **Max-Abs** | `x / max(|x|)` | [-1, 1] | Sparse data |
| **Log transform** | `log(x + 1)` | varies | Right-skewed data |
| **Power transform** | Box-Cox / Yeo-Johnson | varies | Non-normal distributions |

### CRITICAL RULE
> **Fit scalers only on training data. Transform both train and test.**
> Fitting on full data leaks test statistics into training → invalid model!

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 300
df = pd.DataFrame({
    'age':    rng.integers(18, 65, n),
    'income': rng.lognormal(11, 0.5, n),   # right-skewed
    'score':  rng.normal(75, 15, n).clip(0, 100),
    'debt':   rng.exponential(5000, n),
    'target': rng.choice([0, 1], n, p=[0.7, 0.3])
})

# Train/test split FIRST
train_mask = rng.random(n) < 0.8
df_train = df[train_mask].copy()
df_test  = df[~train_mask].copy()
print(f'Train: {len(df_train)}, Test: {len(df_test)}')

In [ ]:
# Z-score Standardization using pandas (manual)
numeric_cols = ['age', 'income', 'score', 'debt']

# Compute ONLY on train
means = df_train[numeric_cols].mean()
stds  = df_train[numeric_cols].std()

df_train_std = (df_train[numeric_cols] - means) / stds
df_test_std  = (df_test[numeric_cols]  - means) / stds   # same train stats!

print('Train stats after standardization:')
print(df_train_std.describe().round(3))

In [ ]:
# Min-Max Normalization (manual)
mins = df_train[numeric_cols].min()
maxs = df_train[numeric_cols].max()

df_train_minmax = (df_train[numeric_cols] - mins) / (maxs - mins)
df_test_minmax  = (df_test[numeric_cols]  - mins) / (maxs - mins)

print('Train range after min-max:')
print(df_train_minmax.describe().loc[['min', 'max']].round(3))

In [ ]:
# Using sklearn scalers (recommended for ML pipelines)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# StandardScaler
ss = StandardScaler()
X_train_ss = ss.fit_transform(df_train[numeric_cols])
X_test_ss  = ss.transform(df_test[numeric_cols])    # only transform!
print('StandardScaler train mean:', X_train_ss.mean(axis=0).round(6))

# RobustScaler (outlier-robust)
rs = RobustScaler()
X_train_rs = rs.fit_transform(df_train[numeric_cols])
X_test_rs  = rs.transform(df_test[numeric_cols])
print('RobustScaler train IQR approx:', (X_train_rs.std(axis=0)).round(3))

In [ ]:
# Log transform for right-skewed income
df_train['log_income'] = np.log1p(df_train['income'])
df_test['log_income']  = np.log1p(df_test['income'])

print('Income distribution before/after log:')
print('Original skew:', df_train['income'].skew().round(3))
print('Log-transformed skew:', df_train['log_income'].skew().round(3))

In [ ]:
# Converting back to DataFrame after sklearn transform
feature_cols = numeric_cols
X_train_scaled_df = pd.DataFrame(X_train_ss, columns=feature_cols,
                                  index=df_train.index)
X_test_scaled_df  = pd.DataFrame(X_test_ss,  columns=feature_cols,
                                  index=df_test.index)
print('Scaled train DataFrame:')
print(X_train_scaled_df.head())